# Free GPU Video Backend — Kaggle / Colab launcher

Runs the `studio/self-hosted-server` from the [gmmarketing](https://github.com/giancolif-maker/gmmarketing) repo
on a free Kaggle or Colab GPU, and exposes it to the internet with a free
Cloudflare quick tunnel (no ngrok signup needed).

## Before you run this

**On Kaggle:** Notebook Settings (right sidebar) → Accelerator → **GPU T4 x2**
(or P100) → Internet → **On**.

**On Colab:** Runtime → Change runtime type → Hardware accelerator → **T4 GPU**.

Then run every cell below in order. The last cell prints a public HTTPS URL —
paste that into the studio app's **Settings → Free / Self-Hosted Server URL**.

**This is a temporary session, not an always-on server.** Kaggle gives you
~30 GPU-hours/week (resets Sundays); Colab is similar but less predictable.
When you're done generating, just stop/close the notebook — there's nothing
to clean up. Next time, re-run this notebook to get a fresh URL (it changes
every time) and update it in Settings again.


In [ ]:
# Sanity check: confirm a GPU is actually attached before we do anything else.
# If this errors or shows "no GPU", go fix the accelerator setting above and restart.
!nvidia-smi


In [ ]:
# Which branch of the repo to pull the server code from.
# Change this to "main" once the Free Mode feature is merged out of its feature branch.
REPO_BRANCH = "claude/open-higgsfield-ai-4bxnm1"
REPO_RAW_BASE = "https://raw.githubusercontent.com/giancolif-maker/gmmarketing"

!mkdir -p /kaggle/working/self-hosted-server
%cd /kaggle/working/self-hosted-server
!curl -fsSL "$REPO_RAW_BASE/$REPO_BRANCH/studio/self-hosted-server/app.py" -o app.py
!curl -fsSL "$REPO_RAW_BASE/$REPO_BRANCH/studio/self-hosted-server/requirements.txt" -o requirements.txt
!echo "--- app.py (first 15 lines) ---" && head -n 15 app.py


In [ ]:
# Install dependencies. This takes a few minutes the first time (torch + diffusers are large).
!pip install -q -r requirements.txt


In [ ]:
# Optional: pick a different open video model. Left as the default (LTX-Video) unless you set this.
import os
os.environ.setdefault("MODEL_ID", "Lightricks/LTX-Video")
print("Using MODEL_ID =", os.environ["MODEL_ID"])


In [ ]:
# Start the FastAPI server in the background.
import subprocess, time, os

server_log = open("server.log", "w")
server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT, env=os.environ.copy(),
)
print(f"Server starting (pid {server.pid})... waiting for it to come up")

import urllib.request
for attempt in range(30):
    time.sleep(2)
    try:
        with urllib.request.urlopen("http://localhost:8000/health", timeout=3) as resp:
            print("Server is up:", resp.read().decode())
            break
    except Exception:
        if attempt == 29:
            print("Server did not come up in time — check server.log below:")
            print(open("server.log").read()[-3000:])
        continue


In [ ]:
# Download a Cloudflare quick tunnel binary (no account/signup needed) and expose port 8000.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("cloudflared downloaded.")


In [ ]:
# Start the tunnel and grab the public URL it prints.
import subprocess, re, threading

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com")

def pump_output():
    global public_url
    for line in tunnel.stdout:
        m = url_pattern.search(line)
        if m and not public_url:
            public_url = m.group(0)

t = threading.Thread(target=pump_output, daemon=True)
t.start()

import time
for _ in range(30):
    time.sleep(1)
    if public_url:
        break

if public_url:
    print("=" * 70)
    print(f"YOUR FREE SERVER URL: {public_url}")
    print("=" * 70)
    print("Paste this into the studio app: Settings -> Free / Self-Hosted Server URL")
    print("Keep this notebook running while you generate videos.")
else:
    print("Didn't catch the URL yet — the tunnel is still starting. Re-run this cell's")
    print("output check, or look at `tunnel.stdout` manually; a *.trycloudflare.com")
    print("line should appear shortly.")


## When you're done

Just stop/close this notebook — nothing needs cleanup, and the tunnel dies
with the session automatically.

If you want to free the GPU without ending the whole notebook:

```python
tunnel.terminate()
server.terminate()
```

## Next time

Re-run this notebook top to bottom. You'll get a **new** `trycloudflare.com`
URL each time — update it in the studio app's Settings again.
